In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "candidates.jsonl"

from src.data_loader import CandidateDataLoader
from src.candidate_processor import CandidateProcessor
from src.semantic_matcher import SemanticMatcher
from src.jd_parser import JDParser
from src.ranking_engine import RankingEngine
from src.reasoning_engine import ReasoningEngine
from src.comparison_engine import ComparisonEngine
from src.report_generator import ReportGenerator

C:\Users\anand\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
loader = CandidateDataLoader(DATA_PATH)
processor = CandidateProcessor()
matcher = SemanticMatcher()

parser = JDParser()

ranker = RankingEngine()
reasoner = ReasoningEngine()
comparer = ComparisonEngine()
reporter = ReportGenerator()

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10304.92it/s]


Model loaded.


In [3]:
candidates = loader.load_candidates()

jd = """
Senior Python Backend Engineer

Requirements:
- Python
- FastAPI
- Docker
- AWS
- PostgreSQL
- 5+ years experience
"""

parsed_jd = parser.parse(jd)

len(candidates)

100000

In [4]:
scores = []

for candidate in candidates[:100]:

    profile = processor.process_profile(candidate)
    skills = processor.process_skills(candidate)

    candidate_text = " ".join([
        profile["headline"],
        profile["summary"],
        skills["skills_text"]
    ])

    semantic = matcher.similarity(
        jd,
        candidate_text
    )

    ranking = ranker.rank(
        semantic,
        profile,
        skills,
        parsed_jd
    )

    scores.append({

        "candidate_id": candidate["candidate_id"],
        "name": profile["name"],

        **ranking

    })

In [5]:
ranked = sorted(
    scores,
    key=lambda x: x["final_score"],
    reverse=True
)

In [6]:
evaluation = reasoner.generate(ranked[0])

evaluation

{'candidate_name': 'Vihaan Naidu',
 'overall_match': 62.69,
 'recommendation': '⭐⭐⭐ Consider',
 'confidence': 'Medium',
 'summary': "The candidate achieved an overall match score of 62.69%. Based on semantic relevance, experience and technical skills, the recommendation is '⭐⭐⭐ Consider'.",
 'strengths': ['Meets required experience',
  'Matched 1 required skill(s): python'],
 'gaps': ['Missing 4 required skill(s): aws, docker, fastapi, sql',
  'Weak semantic alignment with job description'],
 'metrics': {'semantic_score': 55.69,
  'skill_match': 20.0,
  'experience_match': True}}

In [7]:
candidate_1 = ranked[0]
candidate_2 = ranked[1]

comparison = comparer.compare(
    candidate_1,
    candidate_2
)

comparison

{'candidate_1': 'Vihaan Naidu',
 'candidate_2': 'Meera Arora',
 'winner': 'Vihaan Naidu',
 'score_difference': 0.0071,
 'advantages': ['Vihaan Naidu matches more required skills.',
  'Meera Arora has better semantic alignment.'],
 'recommendation': 'Recommend interviewing Vihaan Naidu first.'}

In [8]:
report = reporter.generate(
    evaluation,
    comparison
)

print(report)

AI RECRUITER INTELLIGENCE REPORT

Candidate : Vihaan Naidu
Overall Match : 62.69%
Recommendation : ⭐⭐⭐ Consider
Confidence : Medium

------------------------------------------------------------
SUMMARY
------------------------------------------------------------
The candidate achieved an overall match score of 62.69%. Based on semantic relevance, experience and technical skills, the recommendation is '⭐⭐⭐ Consider'.

STRENGTHS
✓ Meets required experience
✓ Matched 1 required skill(s): python

GAPS
✗ Missing 4 required skill(s): aws, docker, fastapi, sql
✗ Weak semantic alignment with job description

------------------------------------------------------------
COMPARISON
------------------------------------------------------------
Recommend interviewing Vihaan Naidu first.

Reasons:
• Vihaan Naidu matches more required skills.
• Meera Arora has better semantic alignment.



In [9]:
candidate = candidates[0]

skills = processor.process_skills(candidate)

print(skills)

{'skills_list': ['Tailwind', 'NLP', 'Image Classification', 'Fine-tuning LLMs', 'Weights & Biases', 'Speech Recognition', 'Photoshop', 'TTS', 'LoRA', 'Apache Beam', 'AWS', 'Flask', 'BentoML', 'Milvus', 'GANs', 'Statistical Modeling', 'GCP'], 'skills_text': 'Tailwind NLP Image Classification Fine-tuning LLMs Weights & Biases Speech Recognition Photoshop TTS LoRA Apache Beam AWS Flask BentoML Milvus GANs Statistical Modeling GCP', 'num_skills': 17}


In [10]:
for c in candidates:
    profile = processor.process_profile(c)

    if profile["name"] == "Vihaan Naidu":
        print(processor.process_skills(c))
        break

{'skills_list': ['Hadoop', 'JavaScript', 'Databricks', 'Python', 'dbt', 'CI/CD'], 'skills_text': 'Hadoop JavaScript Databricks Python dbt CI/CD', 'num_skills': 6}


In [11]:
print(parsed_jd)

{'raw_text': 'Senior Python Backend Engineer\n\nRequirements:\n- Python\n- FastAPI\n- Docker\n- AWS\n- PostgreSQL\n- 5+ years experience', 'job_title': 'Backend Engineer', 'minimum_experience': 5, 'required_skills': ['Python', 'SQL', 'AWS', 'Docker', 'FastAPI'], 'preferred_skills': [], 'soft_skills': [], 'keywords': ['Docker', 'Python', 'AWS', 'SQL', 'FastAPI']}
